# Home Loan Data Analysis

## Problem Statement
For a safe and secure lending experience, it's important to analyze the past data. In this project, we build a deep learning model to predict the chance of default for future loans using historical data.

**Objective:** Create a model that predicts whether or not an applicant will be able to repay a loan using historical data.

**Domain:** Finance

**Analysis:** Perform data preprocessing and build a deep learning prediction model.

## 1. Import libraries and load data


In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

# Metrics
from sklearn.metrics import (
    classification_report, confusion_matrix,
    recall_score, roc_auc_score, roc_curve
)

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

---
## Step 1: Load the Dataset

In [ ]:
# Load the main dataset
df = pd.read_csv('Home_loan_data.csv')
print(f"Dataset Shape: {df.shape}")
print(f"Number of Rows: {df.shape[0]}")
print(f"Number of Columns: {df.shape[1]}")
df.head()

In [ ]:
# Load the main dataset
df = pd.read_csv('Home_loan_data.csv')
print(f"Dataset Shape: {df.shape}")
print(f"Number of Rows: {df.shape[0]}")
print(f"Number of Columns: {df.shape[1]}")
df.head()

In [ ]:
# Load the data dictionary
data_dict = pd.read_csv('Data_Dictionary.csv')
data_dict.head(10)

In [ ]:
# Basic info about the dataset
print("Dataset Info:")
print("=" * 50)
df.info(verbose=False)

In [ ]:
# Statistical summary of numerical columns
df.describe()

---
## Step 2: Check for Null Values in the Dataset

In [ ]:
# Check null values count and percentage for each column
null_counts = df.isnull().sum()
null_percentage = (df.isnull().sum() / len(df)) * 100

null_df = pd.DataFrame({
    'Null Count': null_counts,
    'Null Percentage (%)': round(null_percentage, 2)
})

# Filter and display only columns with null values
null_df_filtered = null_df[null_df['Null Count'] > 0].sort_values(
    by='Null Percentage (%)', ascending=False
)

print(f"Total columns with null values: {len(null_df_filtered)} out of {df.shape[1]}")
print("=" * 60)
null_df_filtered

In [ ]:
# Visualize top 20 columns with highest null percentages
plt.figure(figsize=(14, 8))
top_null = null_df_filtered.head(20)
plt.barh(top_null.index, top_null['Null Percentage (%)'], color='salmon', edgecolor='darkred')
plt.xlabel('Null Percentage (%)', fontsize=12)
plt.ylabel('Columns', fontsize=12)
plt.title('Top 20 Columns with Highest Null Percentages', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Handle null values
# Strategy:
# 1. Drop columns with more than 40% null values
# 2. Fill numerical columns with median
# 3. Fill categorical columns with mode

threshold = 40
cols_to_drop = null_df_filtered[null_df_filtered['Null Percentage (%)'] > threshold].index.tolist()
print(f"Dropping {len(cols_to_drop)} columns with > {threshold}% null values:")
print(cols_to_drop)

df.drop(columns=cols_to_drop, inplace=True)
print(f"\nDataset shape after dropping columns: {df.shape}")

In [ ]:
# Fill remaining null values
numerical_cols = df.select_dtypes(include=[np.number]).columns
categorical_cols = df.select_dtypes(include=['object']).columns

# Fill numerical with median
for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

# Fill categorical with mode
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

# Verify no null values remain
print(f"Total remaining null values: {df.isnull().sum().sum()}")

---
## Step 3: Print the Percentage of Default to Payer for the TARGET Column

In [ ]:
# TARGET column: 1 = Default (loan not repaid), 0 = Payer (loan repaid)
target_counts = df['TARGET'].value_counts()
target_percentage = df['TARGET'].value_counts(normalize=True) * 100

print("TARGET Column Distribution:")
print("=" * 50)
print(f"\nPayer (TARGET=0)   : {target_counts[0]:,} ({target_percentage[0]:.2f}%)")
print(f"Default (TARGET=1) : {target_counts[1]:,} ({target_percentage[1]:.2f}%)")
print(f"\nRatio of Default to Payer: 1:{target_counts[0]/target_counts[1]:.2f}")
print(f"\n→ The dataset is HIGHLY IMBALANCED with only ~{target_percentage[1]:.1f}% defaults.")

---
## Step 4: Balance the Dataset (Using SMOTE)

In [ ]:
# Prepare features and target
# Drop the SK_ID_CURR (unique identifier) and TARGET column from features
X = df.drop(columns=['SK_ID_CURR', 'TARGET'])
y = df['TARGET']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nData types in features:")
print(X.dtypes.value_counts())

In [ ]:
# Encode categorical columns before SMOTE (SMOTE requires numerical data)
label_encoders = {}
categorical_features = X.select_dtypes(include=['object']).columns

print(f"Categorical columns to encode: {len(categorical_features)}")
print(categorical_features.tolist())

for col in categorical_features:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

print(f"\nAll columns are now numerical: {X.select_dtypes(include=['object']).shape[1] == 0}")

In [ ]:
# Apply SMOTE to balance the dataset
print("Before SMOTE:")
print(y.value_counts())

smote = SMOTE(random_state=42)
X_balanced, y_balanced = smote.fit_resample(X, y)

print(f"\nAfter SMOTE:")
print(pd.Series(y_balanced).value_counts())
print(f"\nBalanced dataset shape: {X_balanced.shape}")

---
## Step 5: Plot the Balanced and Imbalanced Data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Original imbalanced data
colors_imb = ['#2ecc71', '#e74c3c']
original_counts = y.value_counts()
axes[0].bar(['Payer (0)', 'Default (1)'], original_counts.values, color=colors_imb, edgecolor='black')
axes[0].set_title('BEFORE Balancing (Imbalanced)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_xlabel('TARGET', fontsize=12)
for i, v in enumerate(original_counts.values):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold', fontsize=11)

# Plot 2: Balanced data after SMOTE
colors_bal = ['#3498db', '#e67e22']
balanced_counts = pd.Series(y_balanced).value_counts()
axes[1].bar(['Payer (0)', 'Default (1)'], balanced_counts.values, color=colors_bal, edgecolor='black')
axes[1].set_title('AFTER Balancing (SMOTE)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_xlabel('TARGET', fontsize=12)
for i, v in enumerate(balanced_counts.values):
    axes[1].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold', fontsize=11)

plt.suptitle('TARGET Distribution: Before vs After SMOTE', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Pie chart comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].pie(original_counts.values, labels=['Payer (0)', 'Default (1)'],
            autopct='%1.2f%%', colors=colors_imb, startangle=90,
            explode=(0, 0.1), shadow=True, textprops={'fontsize': 12})
axes[0].set_title('Original (Imbalanced)', fontsize=14, fontweight='bold')

axes[1].pie(balanced_counts.values, labels=['Payer (0)', 'Default (1)'],
            autopct='%1.2f%%', colors=colors_bal, startangle=90,
            shadow=True, textprops={'fontsize': 12})
axes[1].set_title('After SMOTE (Balanced)', fontsize=14, fontweight='bold')

plt.suptitle('TARGET Proportion Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Step 6: Encode the Columns Required for the Model

**Note:** Label encoding was already performed in Step 4 (before SMOTE, since SMOTE requires numerical data). Here we scale the features.

In [ ]:
# Verify all columns are encoded (numerical)
print("Data types after encoding:")
print(pd.DataFrame(X_balanced).dtypes.value_counts())
print(f"\nEncoded categorical columns: {list(label_encoders.keys())}")

In [ ]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced, y_balanced, test_size=0.2, random_state=42, stratify=y_balanced
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTraining target distribution:\n{pd.Series(y_train).value_counts()}")
print(f"\nTest target distribution:\n{pd.Series(y_test).value_counts()}")

In [ ]:
# Scale the features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Scaled training data shape: {X_train_scaled.shape}")
print(f"Scaled test data shape: {X_test_scaled.shape}")

---
## Build Deep Learning Model

In [ ]:
# Build the deep learning model
input_dim = X_train_scaled.shape[1]

model = Sequential([
    # Input layer
    Dense(256, activation='relu', input_dim=input_dim),
    BatchNormalization(),
    Dropout(0.3),
    
    # Hidden layer 1
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    
    # Hidden layer 2
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    
    # Hidden layer 3
    Dense(32, activation='relu'),
    Dropout(0.2),
    
    # Output layer
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Train the model with early stopping
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=256,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[1].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Make predictions
y_pred_prob = model.predict(X_test_scaled)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

print("Classification Report:")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=['Payer (0)', 'Default (1)']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Payer (0)', 'Default (1)'],
            yticklabels=['Payer (0)', 'Default (1)'])
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('Actual', fontsize=12)
plt.xlabel('Predicted', fontsize=12)
plt.tight_layout()
plt.show()

---
## Step 7: Calculate Sensitivity (Recall) as a Metric

**Sensitivity (Recall)** = TP / (TP + FN)

It measures the proportion of actual positive cases (defaults) that the model correctly identified.

In [ ]:
---
## Step 8: Calculate the Area Under the ROC Curve (AUC-ROC)

In [ ]:
# Calculate AUC-ROC
auc_score = roc_auc_score(y_test, y_pred_prob)
print(f"AUC-ROC Score: {auc_score:.4f}")

# Calculate ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)

# Plot ROC Curve
plt.figure(figsize=(10, 7))
plt.plot(fpr, tpr, color='#e74c3c', linewidth=2.5, label=f'ROC Curve (AUC = {auc_score:.4f})')
plt.plot([0, 1], [0, 1], color='gray', linewidth=1.5, linestyle='--', label='Random Classifier (AUC = 0.50)')
plt.fill_between(fpr, tpr, alpha=0.15, color='#e74c3c')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=13)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=13)
plt.title('Receiver Operating Characteristic (ROC) Curve', fontsize=15, fontweight='bold')
plt.legend(loc='lower right', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n{'='*50}")
print(f"📊 FINAL MODEL RESULTS SUMMARY")
print(f"{'='*50}")
print(f"Sensitivity (Recall) : {sensitivity:.4f}")
print(f"Specificity          : {specificity:.4f}")
print(f"Precision            : {precision:.4f}")
print(f"AUC-ROC Score        : {auc_score:.4f}")
print(f"{'='*50}")

---
## Conclusion

In this project, we performed the following:

1. **Loaded** the Home Loan dataset (~307K records, 122 features)
2. **Checked for null values** and handled them by dropping high-null columns (>40%) and imputing remaining nulls with median/mode
3. **Analyzed the TARGET distribution** — found the dataset is highly imbalanced (~92% Payers vs ~8% Defaults)
4. **Balanced the dataset** using SMOTE (Synthetic Minority Over-sampling Technique)
5. **Visualized** the data before and after balancing using bar charts and pie charts
6. **Encoded** categorical columns using Label Encoding and scaled features using StandardScaler
7. **Calculated Sensitivity (Recall)** as a key metric for default detection
8. **Calculated and plotted the AUC-ROC curve** to evaluate the model's discriminatory ability

The deep learning model with BatchNormalization and Dropout regularization was trained to predict loan defaults, with sensitivity and AUC-ROC as the primary evaluation metrics.